In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
from dotenv.main import load_dotenv

load_dotenv()
REPO_PATH = os.getenv('REPO_PATH')

sys.path.append(REPO_PATH)

In [2]:
import json

def load_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

def save_json(file_name, file):
    with open(os.path.join(REPO_PATH, "data_files", f"{file_name}.json"), "w", encoding="utf-8") as out_file:
        json.dump(file, out_file, indent=4)

### Havest Wikipedia tables with tables:

In [2]:
from knowledge_maze.create_dataset.create_questions.wikipedia_mining import WikiHarvester
# Initialize Wikipedia harvester
harvester = WikiHarvester(delay=0.05)
# Run Phase 1 - Harvesting titles based on keywords
raw_titles = harvester.fetch_titles()
# Run Phase 2 - Filtering pages for tables
table_titles = harvester.filter_pages_with_tables(raw_titles)

Starting Phase 1: Harvesting titles for 26 keywords...


Harvesting: 100%|█████████████████████████████████████████████████████████████████| 26/26 [02:04<00:00,  4.78s/keyword]



Keyword              | Pages Found
----------------------------------
list of              | 10000     
list                 | 10000     
comparison of        | 693       
rankings of          | 175       
discography of       | 123       
filmography of       | 176       
demographics of      | 1231      
list of countries by | 455       
list of states by    | 13        
list of players      | 93        
list of cities by    | 47        
list of companies by | 7         
list of largest      | 421       
list of most         | 295       
members              | 4459      
teams                | 1222      
countries            | 1422      
cities               | 2263      
species              | 2172      
films                | 7554      
awards               | 10000     
companies            | 1515      
airlines             | 2916      
universities         | 1253      
planets              | 1288      
missions             | 861       
----------------------------------
TOTAL UNIQ

Filtering: 100%|██████████████████████████████████████████████████████████████████| 944/944 [17:03<00:00,  1.08s/batch]


Phase 2 Complete: Kept 29590 pages out of 47177.


In [8]:
save_json("table_titles", table_titles)

In [3]:
table_titles = load_json("C:/Users/Daniel/Desktop/KnowledgeMaze/data_files/table_titles.json")

### Parse and filter Wikipedia tables:

In [4]:
from knowledge_maze.create_dataset.create_questions.table_processing import WikiExtractor
import random
runner = WikiExtractor(max_workers=25)
final_data = runner.bulk_extract(random.sample(table_titles, 1000))

Starting Extraction: 1000 titles | 25 concurrent threads.


Extracting Tables: 100%|████████████████████| 1000/1000 [16:02<00:00,  1.04page/s, Status=RUNNING, Success=717, 429s=0]


FINAL EXTRACTION SUMMARY
------------------------------------------------------------
Total Titles Input:                                | 1000
Pages with Successfully Saved Tables:              | 717
Pages with No Tables Found / Filtered Out:         | 279
Rejected due to Rate Limit (429):                  | 0
! Other Network Errors:                            | 4
------------------------------------------------------------
TOTAL TABLES COLLECTED:                            | 2141
SUCCESS RATE:                                      | 71.7%



In [6]:
final_data[0]

{'table_name': 'List of bean-to-bar chocolate manufacturers > Bean-to-bar chocolate manufacturers',
 'rows': [{'Company': 'Ah Cacao Real Chocolate',
   'Location': 'Mexico',
   'Year founded': 2003,
   'Products': 'Single origin chocolate bars',
   'Notes': 'Artisan producer of dark chocolate and related products made of Mexican cacao. Supports wildlife conservation.'},
  {'Company': 'Amano Artisan Chocolate',
   'Location': 'United States',
   'Year founded': 2006,
   'Products': 'Single origin chocolate (bars, couverture), cocoa nibs',
   'Notes': "Founding member of the Craft Chocolate Makers of America. The Madagascar bar and the Dos Rios Palet d'or have received gold medals from the London Academy of Chocolate."},
  {'Company': 'Amedei',
   'Location': 'Italy',
   'Year founded': 1990,
   'Products': '',
   'Notes': 'Amedei is a luxury chocolate manufacturing company located in Pontedera in the Tuscany region of Italy.'},
  {'Company': 'Askinosie Chocolate',
   'Location': 'United

### Testing:

In [36]:
from knowledge_maze.create_dataset.create_questions.grouping_entities import get_group
try:
    import nest_asyncio
    nest_asyncio.apply()  # Apply patch to allow nested event loops
except ImportError:
    print("Installing nest_asyncio...")


llm_output = get_group(final_data[0])
llm_output

TableGroup(group_description='Bean-to-bar chocolate manufacturers founded 2010-2015', filter_column='Year founded', filter_type='numerical_range', filter_value=[2010, 2015])

In [37]:
import re
import traceback

def filter_table_rows_dict(llm_data, table_rows):
    try:
        # 1. Extract data using Pydantic V2 standards
        if hasattr(llm_data, 'model_dump'):
            data = llm_data.model_dump()
        elif hasattr(llm_data, 'Output') and hasattr(llm_data.Output, 'model_dump'):
            data = llm_data.Output.model_dump()
        elif hasattr(llm_data, 'dict'): # Fallback for V1
            data = llm_data.dict()
        else:
            data = llm_data

        col_name_query = str(data.get('filter_column', "")).lower().strip()
        f_type = str(data.get('filter_type', "")).lower()
        f_val = data.get('filter_value', [])

        if not col_name_query or not table_rows:
            return []

        # 2. Strict Column Matching
        # We look for the BEST match, not just any match
        target_col = None
        for k in table_rows[0].keys():
            # Clean both for a fair comparison
            clean_k = str(k).lower().strip()
            if col_name_query == clean_k or col_name_query in clean_k:
                target_col = k
                break
        
        if target_col is None:
            return []

        filtered_rows = []
        for row in table_rows:
            val_orig = row.get(target_col)
            if val_orig is None: continue
            
            # Sanitize cell content (Handles tuples/lists from scraper)
            if isinstance(val_orig, (tuple, list)) and len(val_orig) > 0:
                cell_str = str(val_orig[0])
            else:
                cell_str = str(val_orig)
            
            cell_str = cell_str.strip()
            if not cell_str: continue

            # --- Strict Filter Logic ---
            if 'list' in f_type:
                # Exact or substring match for list items
                low_cell = cell_str.lower()
                if any(str(item).lower() in low_cell for item in f_val):
                    filtered_rows.append(row)

            elif 'num' in f_type:
                try:
                    # Extract only the first number found (ignores citations like [1])
                    numbers = re.findall(r"[-+]?\d*\.\d+|\d+", cell_str.replace(',', ''))
                    if not numbers: continue
                    
                    val_as_float = float(numbers[0])
                    low_bound = float(f_val[0])
                    high_bound = float(f_val[1])
                    
                    # Core check
                    if low_bound <= val_as_float <= high_bound:
                        filtered_rows.append(row)
                except:
                    continue

            elif 'temp' in f_type:
                try:
                    # Look for 4-digit years
                    year_match = re.search(r'\b(1\d{3}|20\d{2})\b', cell_str)
                    if year_match:
                        row_year = int(year_match.group(0))
                        # Handle date strings or years in f_val
                        start_y = int(str(f_val[0])[:4])
                        end_y = int(str(f_val[1])[:4])
                        if start_y <= row_year <= end_y:
                            filtered_rows.append(row)
                except:
                    continue

        return filtered_rows

    except Exception as e:
        print(f"CRITICAL FILTER ERROR: {e}")
        traceback.print_exc()
        return []

In [39]:
# After your LLM call
filtered_data = filter_table_rows_dict(llm_output, final_data[0]['rows'])

In [42]:
import re

def extract_entity_list(llm_data, table_rows):
    try:
        # Pydantic V2 migration safe
        data = llm_data.model_dump() if hasattr(llm_data, 'model_dump') else llm_data
        
        ent_col_query = str(data.get('entity_column', "")).lower().strip()
        fil_col_query = str(data.get('filter_column', "")).lower().strip()
        f_type = data.get('filter_type')
        f_val = data.get('filter_value')
        sort_order = data.get('sort_order', 'desc')

        if not table_rows: return []

        # Find the actual column keys
        keys = table_rows[0].keys()
        entity_key = next((k for k in keys if ent_col_query in str(k).lower()), None)
        filter_key = next((k for k in keys if fil_col_query in str(k).lower()), None)

        if not entity_key or not filter_key: return []

        # --- STEP 1: Numerical Cleaning for Sorting/Filtering ---
        processed_data = []
        for row in table_rows:
            raw_ent = row.get(entity_key, "Unknown")
            raw_fil = row.get(filter_key, 0)
            
            # Extract name (handle tuples/lists from scraper)
            name = str(raw_ent[0] if isinstance(raw_ent, (list, tuple)) else raw_ent).strip()
            
            # Extract number for sorting/filtering
            clean_fil = re.sub(r'[^\d.]', '', str(raw_fil))
            val_as_float = float(clean_num[0]) if (clean_num := re.findall(r"[-+]?\d*\.\d+|\d+", str(raw_fil).replace(',', ''))) else 0.0
            
            processed_data.append({"name": name, "val": val_as_float, "original_row": row})

        # --- STEP 2: Apply Logic ---
        result_entities = []

        if f_type == "top_n":
            # Sort by the numerical value
            sorted_data = sorted(processed_data, key=lambda x: x['val'], reverse=(sort_order == 'desc'))
            # Take exactly N
            limit = min(max(int(f_val), 3), 10) # Safety clamp between 3 and 10
            result_entities = [d['name'] for d in sorted_data[:limit]]

        elif f_type == "numerical_range":
            # Traditional slice
            low, high = float(f_val[0]), float(f_val[1])
            matches = [d['name'] for d in processed_data if low <= d['val'] <= high]
            result_entities = matches[:10] # Ensure we don't exceed 10

        return result_entities

    except Exception as e:
        print(f"Extraction Error: {e}")
        return []

In [47]:
final_data[2]

{'table_name': 'List of Spanish films of 2017 > Box office',
 'rows': [{'Rank': 1,
   'Title': 'Tad the Lost Explorer and the Secret of King Midas (Tadeo Jones 2: El secreto del rey Midas)',
   'Distributor': 'Paramount Pictures',
   'Admissions': 3167110,
   'Domestic gross (€)': 17553149},
  {'Rank': 2,
   'Title': 'Perfect Strangers (Perfectos desconocidos)',
   'Distributor': 'Universal Pictures',
   'Admissions': 2020940,
   'Domestic gross (€)': 12969658},
  {'Rank': 3,
   'Title': "It's for Your Own Good (Es por tu bien)",
   'Distributor': 'Buena Vista International',
   'Admissions': 1584578,
   'Domestic gross (€)': 9638993},
  {'Rank': 4,
   'Title': 'Marrowbone (El secreto de Marrowbone)',
   'Distributor': 'Universal Pictures',
   'Admissions': 1158521,
   'Domestic gross (€)': 7153428},
  {'Rank': 5,
   'Title': 'Lord, Give Me Patience (Señor, dame paciencia)',
   'Distributor': 'Warner Bros. Pictures',
   'Admissions': 1106441,
   'Domestic gross (€)': 6682652},
  {'Rank

In [54]:
llm_output = get_group(final_data[0])
llm_output

TableGroup(group_description='Latest 7 chocolate manufacturers by founding date', entity_column='Company', filter_column='Year founded', filter_type='top_n', filter_value=7, sort_order='desc')

In [49]:
extract_entity_list(llm_output, final_data[2]['rows'])

['Tad the Lost Explorer and the Secret of King Midas (Tadeo Jones 2: El secreto del rey Midas)',
 'Perfect Strangers (Perfectos desconocidos)',
 "It's for Your Own Good (Es por tu bien)",
 'Marrowbone (El secreto de Marrowbone)',
 'Lord, Give Me Patience (Señor, dame paciencia)',
 'Toc Toc',
 'The Invisible Guardian (El guardián invisible)']

In [51]:
import re

def extract_grouped_data(llm_data, table_rows):
    try:
        # Pydantic V2 migration safe
        data = llm_data.model_dump() if hasattr(llm_data, 'model_dump') else llm_data
        
        group_desc = data.get('group_description', "")
        ent_col_query = str(data.get('entity_column', "")).lower().strip()
        fil_col_query = str(data.get('filter_column', "")).lower().strip()
        f_type = data.get('filter_type')
        f_val = data.get('filter_value')
        sort_order = data.get('sort_order', 'desc')

        if not table_rows: return {}

        # 1. Identify the actual keys in the original dictionary
        keys = list(table_rows[0].keys())
        entity_key = next((k for k in keys if ent_col_query in str(k).lower()), None)
        filter_key = next((k for k in keys if fil_col_query in str(k).lower()), None)

        if not entity_key or not filter_key: return {}

        # 2. Pre-process and Clean data
        processed_pool = []
        for row in table_rows:
            raw_ent = row.get(entity_key, "Unknown")
            raw_fil = row.get(filter_key, 0)
            
            # Clean entity name
            name = str(raw_ent[0] if isinstance(raw_ent, (list, tuple)) else raw_ent).strip()
            
            # Clean numerical value for sorting/filtering
            # Keep original raw_fil for the "Group entities" value output
            clean_num_list = re.findall(r"[-+]?\d*\.\d+|\d+", str(raw_fil).replace(',', ''))
            val_as_float = float(clean_num_list[0]) if clean_num_list else 0.0
            
            processed_pool.append({
                "name": name, 
                "numeric_val": val_as_float, 
                "display_val": raw_fil, # This is the "Y" in your entity:Y dict
                "original_row": row.copy() # Copy to avoid mutating original source
            })

        # 3. Filter/Sort Logic
        selected_matches = []
        if f_type == "top_n":
            sorted_pool = sorted(processed_pool, key=lambda x: x['numeric_val'], reverse=(sort_order == 'desc'))
            limit = min(max(int(f_val), 3), 10)
            selected_matches = sorted_pool[:limit]
        
        elif f_type == "numerical_range":
            low, high = float(f_val[0]), float(f_val[1])
            selected_matches = [d for d in processed_pool if low <= d['numeric_val'] <= high][:10]

        # 4. Construct Final Output Dict
        group_entities = {}
        group_data = []

        for match in selected_matches:
            # Add to entity:value dict
            group_entities[match['name']] = match['display_val']
            
            # Prepare row data: Remove the filter column
            pruned_row = match['original_row']
            if filter_key in pruned_row:
                del pruned_row[filter_key]
            
            group_data.append(pruned_row)

        return {
            "Group description": group_desc,
            "Group entities": group_entities,
            "Group data": group_data
        }

    except Exception as e:
        print(f"Extraction Error: {e}")
        return {}

In [55]:
extract_grouped_data(llm_output, final_data[0]['rows'])

{'Group description': 'Latest 7 chocolate manufacturers by founding date',
 'Group entities': {'Q sweet': 2019,
  'Fu Wan Chocolate': 2015,
  'Choco Togo': 2014,
  'Chocolala': 2013,
  'Castronovo Chocolate': 2012,
  'Malagos Chocolate': 2012,
  'Rick Jordan Chocolatier': 2011},
 'Group data': [{'Company': 'Q sweet',
   'Location': 'Taiwan',
   'Products': 'Chocolate truffles, bars, gift boxes',
   'Notes': ''},
  {'Company': 'Fu Wan Chocolate',
   'Location': 'Taiwan',
   'Products': 'Single-origin dark chocolates, Fermented and specialty bars combining cocoa with local flavors (e.g., lychee, tea, shrimp), Drinking chocolate, Limited editions based on Taiwanese terroir',
   'Notes': ''},
  {'Company': 'Choco Togo',
   'Location': 'Togo',
   'Products': 'Chocolate bars, hot chocolate, chocolate cake, and other chocolate products',
   'Notes': 'The first chocolate manufacturer in Togo.'},
  {'Company': 'Chocolala',
   'Location': 'Estonia',
   'Products': 'Single origin chocolate bars (